# Pandas Practice: Describing Data & Creating New Variables
**Dataset:** a simulated behavioral study (reaction-time task + questionnaires).

**How to use this notebook**
1. Run the setup cells to generate the data.
2. Read each *Worked example*, then try the **Your turn** exercises (cells with `# TODO`).
3. Solutions are at the bottom — try first!

**Part 1** Explore & describe the data  
**Part 2** Create new variables  
**Part 3** Mini-project

## Setup: build the behavior database

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200

df = pd.DataFrame({
    "participant_id": [f"P{i:03d}" for i in range(1, n + 1)],
    "age": rng.integers(18, 70, n),
    "gender": rng.choice(["Female", "Male", "Non-binary"], n, p=[0.48, 0.48, 0.04]),
    "condition": rng.choice(["Control", "Treatment"], n),
    "session_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "sleep_hours": rng.normal(6.8, 1.2, n).round(1),
    "anxiety_1": rng.integers(1, 6, n),   # 1-5 Likert
    "anxiety_2": rng.integers(1, 6, n),
    "anxiety_3_rev": rng.integers(1, 6, n),  # reverse-worded item
    "mean_rt_ms": rng.normal(520, 90, n).round(0),
    "n_trials": 100,
    "n_correct": rng.integers(60, 100, n),
})

# Treatment slightly faster; add some realistic mess
df.loc[df["condition"] == "Treatment", "mean_rt_ms"] -= 25
df.loc[rng.choice(n, 12, replace=False), "sleep_hours"] = np.nan
df.loc[rng.choice(n, 5, replace=False), "mean_rt_ms"] = np.nan
df.loc[[7, 88], "age"] = [199, 5]          # impossible values (data entry errors)
df.head()

# Part 1 — Describing the data

### 1.1 First look: shape, columns, types

In [ ]:
print(df.shape)          # (rows, columns)
print(df.columns.tolist())
df.dtypes

In [ ]:
df.info()

### 1.2 Summary statistics

In [ ]:
df.describe()                      # numeric columns

In [ ]:
df.describe(include="object")      # categorical columns

In [ ]:
df.describe(percentiles=[.1, .25, .5, .75, .9]).T   # transpose = easier to read

### 1.3 Categorical variables: counts and proportions

In [ ]:
print(df["condition"].value_counts())
print(df["gender"].value_counts(normalize=True).round(3))

### 1.4 Missing data

In [ ]:
df.isna().sum()

In [ ]:
(df.isna().mean() * 100).round(1)   # % missing per column

### 1.5 Group comparisons

In [ ]:
df.groupby("condition")["mean_rt_ms"].agg(["count", "mean", "std", "min", "max"]).round(1)

In [ ]:
df.groupby(["condition", "gender"]).agg(
    n=("participant_id", "count"),
    mean_rt=("mean_rt_ms", "mean"),
    mean_sleep=("sleep_hours", "mean"),
).round(1)

### 1.6 Spotting outliers / impossible values

In [ ]:
df[(df["age"] < 18) | (df["age"] > 100)]

In [ ]:
df["mean_rt_ms"].quantile([0, .01, .25, .5, .75, .99, 1])

In [ ]:
# Correlations between numeric variables
df[["age", "sleep_hours", "mean_rt_ms", "n_correct"]].corr().round(2)

### ✏️ Your turn — Part 1
1. How many participants are in each `condition` **and** `gender` combination? (hint: `pd.crosstab`)
2. What is the median `sleep_hours`? What is its standard deviation?
3. Which column has the most missing values?
4. What is the mean `n_correct` for people aged 40 or older?
5. Find the 5 slowest participants (largest `mean_rt_ms`).

In [ ]:
# 1.
# TODO

# 2.
# TODO

# 3.
# TODO

# 4.
# TODO

# 5.
# TODO

# Part 2 — Creating new variables

First, clean the impossible ages. Work on a **copy** so the original stays intact.

In [ ]:
d = df.copy()
d.loc[(d["age"] < 18) | (d["age"] > 100), "age"] = np.nan
d["age"].describe()

### 2.1 Simple arithmetic: accuracy proportion

In [ ]:
d["accuracy"] = d["n_correct"] / d["n_trials"]
d["pct_correct"] = d["accuracy"] * 100
d[["participant_id", "n_correct", "accuracy"]].head()

### 2.2 Binary variable with `np.where`

In [ ]:
d["short_sleeper"] = np.where(d["sleep_hours"] < 6, 1, 0)   # note: NaN becomes 0 -- be careful!
d["short_sleeper"].value_counts()

In [ ]:
# Safer: keep missing values as missing
d["short_sleeper"] = np.where(d["sleep_hours"].isna(), np.nan,
                              np.where(d["sleep_hours"] < 6, 1, 0))
d["short_sleeper"].value_counts(dropna=False)

### 2.3 Categories from numbers: `pd.cut` and `pd.qcut`

In [ ]:
# Fixed bins
d["age_group"] = pd.cut(d["age"], bins=[17, 29, 44, 59, 100],
                        labels=["18-29", "30-44", "45-59", "60+"])
print(d["age_group"].value_counts().sort_index())

# Quantile bins (equal-sized groups)
d["rt_tertile"] = pd.qcut(d["mean_rt_ms"], q=3, labels=["Fast", "Medium", "Slow"])
print(d["rt_tertile"].value_counts())

### 2.4 Multiple conditions: `np.select`

In [ ]:
conditions = [
    (d["accuracy"] >= 0.90),
    (d["accuracy"] >= 0.75),
]
choices = ["High", "Medium"]
d["performance"] = np.select(conditions, choices, default="Low")
d["performance"].value_counts()

### 2.5 Reverse-coding & composite scores

In [ ]:
# Reverse-code a 1-5 item: new = (max + min) - old = 6 - old
d["anxiety_3"] = 6 - d["anxiety_3_rev"]

# Composite score: mean of the three items
d["anxiety_score"] = d[["anxiety_1", "anxiety_2", "anxiety_3"]].mean(axis=1)
d[["anxiety_1", "anxiety_2", "anxiety_3_rev", "anxiety_3", "anxiety_score"]].head()

### 2.6 Standardizing (z-scores) and centering

In [ ]:
d["rt_z"] = (d["mean_rt_ms"] - d["mean_rt_ms"].mean()) / d["mean_rt_ms"].std()
d["age_centered"] = d["age"] - d["age"].mean()
d[["mean_rt_ms", "rt_z"]].describe().round(2)

### 2.7 Group-wise variables with `transform`

In [ ]:
# Each person's RT relative to their own condition's mean
d["rt_cond_mean"] = d.groupby("condition")["mean_rt_ms"].transform("mean")
d["rt_diff_from_cond"] = d["mean_rt_ms"] - d["rt_cond_mean"]

# Rank within condition (1 = fastest)
d["rt_rank_in_cond"] = d.groupby("condition")["mean_rt_ms"].rank(method="min")
d[["participant_id", "condition", "mean_rt_ms", "rt_cond_mean", "rt_diff_from_cond", "rt_rank_in_cond"]].head()

### 2.8 Dates

In [ ]:
d["month"] = d["session_date"].dt.month_name()
d["weekday"] = d["session_date"].dt.day_name()
d["is_weekend"] = d["session_date"].dt.dayofweek >= 5
d["days_since_start"] = (d["session_date"] - d["session_date"].min()).dt.days
d[["session_date", "month", "weekday", "is_weekend", "days_since_start"]].head()

### 2.9 Recoding labels with `map` and custom functions with `apply`

In [ ]:
d["condition_code"] = d["condition"].map({"Control": 0, "Treatment": 1})

def flag_valid(row):
    # a participant is valid if RT and sleep are present and accuracy is above chance (50%)
    return (pd.notna(row["mean_rt_ms"])) and (pd.notna(row["sleep_hours"])) and (row["accuracy"] > 0.5)

d["valid"] = d.apply(flag_valid, axis=1)
d["valid"].value_counts()

### 2.10 Other handy tools: `assign`, `rename`, `drop`, `clip`, `fillna`

In [ ]:
d = (d
     .assign(sleep_filled=lambda x: x["sleep_hours"].fillna(x["sleep_hours"].median()),
             rt_clipped=lambda x: x["mean_rt_ms"].clip(lower=300, upper=800))
     .rename(columns={"mean_rt_ms": "rt_ms"})
     .drop(columns=["anxiety_3_rev"]))
d.columns.tolist()

### ✏️ Your turn — Part 2
Using `d`:
1. Create `rt_seconds` = `rt_ms` / 1000.
2. Create `well_rested` = `True` if `sleep_hours` ≥ 7, else `False`.
3. Create `sleep_category` using `pd.cut`: "Low" (<6), "Normal" (6–8), "High" (>8).
4. Create `anxiety_level` with `np.select`: "Low" (<2.5), "Moderate" (2.5–3.5), "High" (>3.5).
5. Create `speed_accuracy` = `accuracy` / `rt_seconds` (a rough efficiency index).
6. Create `age_z`: z-score of age **within each gender** (use `groupby` + `transform`).
7. Create `is_summer` = `True` if the session was in June, July or August.

In [ ]:
# 1.
# TODO

# 2.
# TODO

# 3.
# TODO

# 4.
# TODO

# 5.
# TODO

# 6.
# TODO

# 7.
# TODO

# Part 3 — Mini-project

Answer with code, using the new variables you built:

1. What % of participants are `valid`?
2. Among valid participants, compare mean `rt_ms` and `accuracy` between Control and Treatment.
3. Which `age_group` × `performance` combination is most common? (`pd.crosstab` with `normalize="index"`)
4. Is `anxiety_score` correlated with `rt_ms` and `accuracy`?
5. Make a clean final dataframe `final` containing only valid participants and the columns:
   `participant_id, condition, age_group, sleep_category, anxiety_score, rt_ms, accuracy, performance`. Save it with `final.to_csv("final.csv", index=False)`.

In [ ]:
# TODO

---
# Solutions

### Part 1

In [ ]:
# 1.
print(pd.crosstab(df["condition"], df["gender"]))
# 2.
print(df["sleep_hours"].median(), df["sleep_hours"].std())
# 3.
print(df.isna().sum().idxmax(), df.isna().sum().max())
# 4.
print(df.loc[df["age"] >= 40, "n_correct"].mean())
# 5.
df.nlargest(5, "mean_rt_ms")[["participant_id", "mean_rt_ms"]]

### Part 2

In [ ]:
d["rt_seconds"] = d["rt_ms"] / 1000
d["well_rested"] = d["sleep_hours"] >= 7          # NaN -> False; use .where to keep NaN if needed
d["sleep_category"] = pd.cut(d["sleep_hours"], bins=[0, 6, 8, 24], labels=["Low", "Normal", "High"], right=False)
d["anxiety_level"] = np.select(
    [d["anxiety_score"] < 2.5, d["anxiety_score"] <= 3.5],
    ["Low", "Moderate"], default="High")
d["speed_accuracy"] = d["accuracy"] / d["rt_seconds"]
d["age_z"] = d.groupby("gender")["age"].transform(lambda s: (s - s.mean()) / s.std())
d["is_summer"] = d["session_date"].dt.month.isin([6, 7, 8])
d[["rt_seconds", "well_rested", "sleep_category", "anxiety_level", "speed_accuracy", "age_z", "is_summer"]].head()

### Part 3

In [ ]:
# 1.
print((d["valid"].mean() * 100).round(1), "% valid")

# 2.
v = d[d["valid"]]
print(v.groupby("condition")[["rt_ms", "accuracy"]].mean().round(3))

# 3.
print(pd.crosstab(d["age_group"], d["performance"], normalize="index").round(2))

# 4.
print(v[["anxiety_score", "rt_ms", "accuracy"]].corr().round(2))

# 5.
final = v[["participant_id", "condition", "age_group", "sleep_category",
           "anxiety_score", "rt_ms", "accuracy", "performance"]].copy()
final.to_csv("final.csv", index=False)
final.head()

## Cheat sheet
| Goal | Tool |
|---|---|
| Quick overview | `df.info()`, `df.describe()`, `df.head()` |
| Counts / proportions | `value_counts(normalize=True)`, `pd.crosstab` |
| Group summaries | `groupby().agg()` |
| Arithmetic variable | `df["c"] = df["a"] / df["b"]` |
| Two-way split | `np.where(cond, a, b)` |
| Many-way split | `np.select(conds, choices, default)` |
| Numeric → bins | `pd.cut` (fixed), `pd.qcut` (quantiles) |
| Recode values | `.map({...})` |
| Row-wise logic | `.apply(func, axis=1)` |
| Group-level stat per row | `groupby().transform()` |
| Dates | `.dt.month`, `.dt.dayofweek`, date subtraction |